In [1]:
import numpy as np
import librosa
from scipy.ndimage import median_filter
from collections import defaultdict

import os
import glob
from pathlib import Path
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
class TemplateChordRecognizer:
    def __init__(self, sample_rate=44100):
        self.sr = sample_rate
        # 1. Define Frame Settings based on the paper
        # Frame length = 753 ms, Hop size = 93 ms 
        self.n_fft = int(0.753 * self.sr)
        self.hop_length = int(0.093 * self.sr)
        
        # 2. Generate Chord Templates
        # We use the Single Harmonic Model (1 harmonic) [cite: 126]
        # We include Major, Minor, and Dominant Seventh types 
        self.templates, self.chord_labels = self._generate_templates()

    def _generate_templates(self):
        """
        Generates 24 templates (12 roots * 2 types: Maj, Min).
        Templates are binary masks normalized so the sum of amplitudes is 1.
        """
        templates = []
        labels = []
        
        # Base pitch indices for chord types relative to root C (index 0)
        # Major: Root, Major 3rd (4), Perfect 5th (7)
        maj_intervals = [0, 4, 7]
        # Minor: Root, Minor 3rd (3), Perfect 5th (7)
        min_intervals = [0, 3, 7]     

        definitions = {
            'maj': maj_intervals,
            'min': min_intervals,
        }
        
        chroma_notes = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']

        # Generate for all 12 semitones
        for root_idx, root_name in enumerate(chroma_notes):
            for chord_type, intervals in definitions.items():
                # Create empty 12-dim vector
                template = np.zeros(12)
                
                # Set 1s for notes present in the chord (cyclic shift)
                for interval in intervals:
                    note_idx = (root_idx + interval) % 12
                    template[note_idx] = 1.0
                
                # Normalize so sum is 1 [cite: 88]
                template /= np.sum(template)
                
                templates.append(template)
                labels.append(f"{root_name}:{chord_type}")
                
        return np.array(templates), labels

    def _kl_divergence_v2(self, p, c, epsilon=1e-9):
        """
        Calculates KL2 Divergence: D(Template || Chroma).
        p: Chord Template (Reference)
        c: Chroma Vector (Observation)
        
        Note: The paper specifies a scaling parameter h. 
        Mathematically, minimizing KL divergence D(p || h*c) results in 
        normalizing c such that sum(c) == sum(p). 
        Since sum(p)=1, we effectively just normalize c to sum to 1.
        """
        # Avoid division by zero or log(0)
        c = c + epsilon
        p = p + epsilon
        
        # Normalize chroma vector to sum to 1 (Implicit calculation of h)
        c_norm = c / np.sum(c)
        
        # KL Divergence formula: sum(p * log(p / c))
        # We perform this for the single template p against the specific chroma frame c
        return np.sum(p * np.log(p / c_norm))

    def predict(self, audio_path):
            # 1. Load Audio
            y, _ = librosa.load(audio_path, sr=self.sr)
            
            # 2. Extract Features (Chromagram)
            # We ensure the frame specifications match the paper
            chromagram = librosa.feature.chroma_stft(
                y=y, sr=self.sr, n_fft=self.n_fft, hop_length=self.hop_length
            )
            
            # Shape: (12, N_frames) -> Transpose to (N_frames, 12) for iteration
            chromagram = chromagram.T
            
            # 3. Calculate Measures of Fit (Distance Matrix)
            n_frames = chromagram.shape[0]
            n_templates = len(self.templates)
            distance_matrix = np.zeros((n_frames, n_templates))
            
            for t_idx, template in enumerate(self.templates):
                for f_idx in range(n_frames):
                    frame_chroma = chromagram[f_idx]
                    
                    # Skip silence/empty frames to avoid errors
                    if np.sum(frame_chroma) == 0:
                        distance_matrix[f_idx, t_idx] = np.inf
                    else:
                        # Calculate KL2 distance
                        distance_matrix[f_idx, t_idx] = self._kl_divergence_v2(template, frame_chroma)

            # 4. Post-processing Filtering 
            filtered_distances = median_filter(distance_matrix, size=(17, 1))
            
            # 5. Chord Detection
            # The detected chord is the one minimizing the fit measure
            chord_indices = np.argmin(filtered_distances, axis=1)
            
            # Map indices to labels
            detected_chords = [self.chord_labels[idx] for idx in chord_indices]
            
            # Generate timestamps
            times = librosa.frames_to_time(np.arange(n_frames), sr=self.sr, hop_length=self.hop_length)
            
            return list(zip(times, detected_chords))
    
    def predict_single(self, audio_path):
        """
        Predicts a single chord for the entire audio file using duration-weighted voting.
        
        Returns:
            str: The chord label that was detected for the longest total duration.
        """
        frame_predictions = self.predict(audio_path)
        
        if not frame_predictions:
            return None
        
        # Calculate duration for each frame by looking at time differences
        chord_durations = defaultdict(float)
        
        for i, (time, chord) in enumerate(frame_predictions):
            # Calculate the duration this frame represents
            if i < len(frame_predictions) - 1:
                # Duration is time until next frame
                duration = frame_predictions[i + 1][0] - time
            else:
                # For last frame, use the hop_length duration
                duration = self.hop_length / self.sr
            
            chord_durations[chord] += duration
        
        # Return the chord with the maximum total duration
        best_chord = max(chord_durations, key=chord_durations.get)
        return best_chord

In [3]:
def folder_to_model_label(folder_name: str) -> str:
    """
    Convert folder name (e.g., 'C', 'Cm', 'C#', 'C#m') to model label format (e.g., 'C:maj', 'C:min').
    
    Note: The model only supports maj, min, and 7 (dominant 7th).
    Since the dataset only has major and minor chords, we map accordingly.
    """
    if folder_name.endswith('m'):
        # Minor chord: remove 'm' suffix
        root = folder_name[:-1]
        return f"{root}:min"
    else:
        # Major chord
        return f"{folder_name}:maj"

In [4]:
def model_label_to_simple(label: str) -> str:
    """
    Convert model label (e.g., 'C:maj', 'C:min') back to simple format (e.g., 'C', 'Cm').
    Useful for display purposes.
    """
    if ':min' in label:
        root = label.replace(':min', '')
        return f"{root}m"
    elif ':maj' in label:
        return label.replace(':maj', '')
    return label

In [5]:
def get_test_files(test_dir: str) -> list:
    """
    Collect all test audio files with their ground truth labels.
    
    Returns:
        List of tuples: (file_path, ground_truth_model_label, folder_name)
    """
    test_files = []
    test_path = Path(test_dir)
    
    for chord_folder in test_path.iterdir():
        if chord_folder.is_dir():
            folder_name = chord_folder.name
            model_label = folder_to_model_label(folder_name)
            
            # Get all .wav files in this folder
            wav_files = list(chord_folder.glob("*.wav"))
            for wav_file in wav_files:
                test_files.append((str(wav_file), model_label, folder_name))
    
    return test_files

In [6]:
def evaluate_model(test_dir: str, output_path = str,save_confusion_matrix: bool = True):
    """
    Evaluate the TemplateChordRecognizer on the test dataset.
    
    Args:
        test_dir: Path to the test directory containing chord subfolders
        save_confusion_matrix: Whether to save a confusion matrix plot
    """
    print("=" * 60)
    print("Template-Based Chord Recognition - Model Evaluation")
    print("=" * 60)
    
    # Initialize the recognizer
    print("\nInitializing TemplateChordRecognizer...")
    recognizer = TemplateChordRecognizer()
    
    # Collect test files
    print(f"\nCollecting test files from: {test_dir}")
    test_files = get_test_files(test_dir)
    print(f"Found {len(test_files)} test samples")
    
    # Count samples per class
    class_counts = {}
    for _, label, folder in test_files:
        class_counts[folder] = class_counts.get(folder, 0) + 1
    
    print("\nSamples per class:")
    for chord in sorted(class_counts.keys()):
        print(f"  {chord}: {class_counts[chord]}")
    
    # Run predictions
    print("\nRunning predictions...")
    y_true = []
    y_pred = []
    results_detail = []
    
    for file_path, ground_truth, folder_name in tqdm(test_files, desc="Classifying"):
        try:
            prediction = recognizer.predict_single(file_path)
            y_true.append(ground_truth)
            y_pred.append(prediction)
            
            is_correct = prediction == ground_truth
            results_detail.append({
                'file': os.path.basename(file_path),
                'folder': folder_name,
                'ground_truth': ground_truth,
                'prediction': prediction,
                'correct': is_correct
            })
            
        except Exception as e:
            print(f"\nError processing {file_path}: {e}")
            continue
    
    # Calculate metrics
    print("\n" + "=" * 60)
    print("EVALUATION RESULTS")
    print("=" * 60)
    
    # Accuracy
    accuracy = accuracy_score(y_true, y_pred)
    print(f"\nAccuracy: {accuracy:.4f} ({accuracy * 100:.2f}%)")
    
    # F1 Scores
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    f1_weighted = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    print(f"F1 Score (Macro):    {f1_macro:.4f}")
    print(f"F1 Score (Weighted): {f1_weighted:.4f}")
    
    # Get unique labels for the report
    all_labels = sorted(set(y_true + y_pred))
    
    # Classification Report
    print("\n" + "-" * 60)
    print("CLASSIFICATION REPORT")
    print("-" * 60)
    print(classification_report(y_true, y_pred, labels=all_labels, zero_division=0))
    
    # Show some incorrect predictions
    incorrect = [r for r in results_detail if not r['correct']]
    if incorrect:
        print("\n" + "-" * 60)
        print(f"SAMPLE MISCLASSIFICATIONS ({len(incorrect)} total errors)")
        print("-" * 60)
        for r in incorrect[:15]:  # Show first 15 errors
            print(f"  {r['folder']:5s} -> {model_label_to_simple(r['prediction']):5s} | {r['file']}")
        if len(incorrect) > 15:
            print(f"  ... and {len(incorrect) - 15} more errors")
    
    # Confusion Matrix
    if save_confusion_matrix:
        print("\nGenerating confusion matrix...")
        
        # Create simplified labels for display
        display_labels = [model_label_to_simple(l) for l in all_labels]
        
        cm = confusion_matrix(y_true, y_pred, labels=all_labels)
        
        plt.figure(figsize=(16, 14))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=display_labels,
                    yticklabels=display_labels)
        plt.xlabel('Predicted')
        plt.ylabel('True')
        plt.title(f'Chord Recognition Confusion Matrix\nAccuracy: {accuracy:.2%} | F1 (Macro): {f1_macro:.4f}')
        plt.tight_layout()
        
        plt.show
        plt.savefig(os.path.join(output_path, 'patter_matching_confusion_matrix.png'), dpi=150)
        print(f"Confusion matrix saved to: {output_path}")
        plt.close()
    
    # Summary
    print("\n" + "=" * 60)
    print("SUMMARY")
    print("=" * 60)
    print(f"Total samples:     {len(y_true)}")
    print(f"Correct:           {sum(1 for t, p in zip(y_true, y_pred) if t == p)}")
    print(f"Incorrect:         {sum(1 for t, p in zip(y_true, y_pred) if t != p)}")
    print(f"Accuracy:          {accuracy:.2%}")
    print(f"F1 Score (Macro):  {f1_macro:.4f}")
    print("=" * 60)
    
    return {
        'accuracy': accuracy,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
        'y_true': y_true,
        'y_pred': y_pred,
        'results_detail': results_detail
    }

In [7]:
test_dir = '/kaggle/input/guitarset-chord-recognition/chords/test'
output_dir = '/kaggle/working/'

results = evaluate_model(test_dir, output_dir)

Template-Based Chord Recognition - Model Evaluation

Initializing TemplateChordRecognizer...

Found 254 test samples

Samples per class:
  A: 11
  A#: 11
  A#m: 12
  Am: 6
  B: 11
  Bm: 8
  C: 12
  C#: 12
  C#m: 8
  Cm: 5
  D: 11
  D#: 12
  D#m: 8
  Dm: 9
  E: 12
  Em: 9
  F: 15
  F#: 11
  F#m: 6
  Fm: 15
  G: 14
  G#: 12
  G#m: 12
  Gm: 12

Running predictions...


Classifying: 100%|██████████| 254/254 [01:00<00:00,  4.17it/s]



EVALUATION RESULTS

Accuracy: 0.6024 (60.24%)
F1 Score (Macro):    0.5780
F1 Score (Weighted): 0.5976

------------------------------------------------------------
CLASSIFICATION REPORT
------------------------------------------------------------
              precision    recall  f1-score   support

      A#:maj       1.00      0.64      0.78        11
      A#:min       0.88      0.58      0.70        12
       A:maj       0.67      0.36      0.47        11
       A:min       0.62      0.83      0.71         6
       B:maj       1.00      0.45      0.62        11
       B:min       0.33      0.25      0.29         8
      C#:maj       0.64      0.58      0.61        12
      C#:min       0.50      0.25      0.33         8
       C:maj       0.71      0.83      0.77        12
       C:min       0.50      0.20      0.29         5
      D#:maj       0.55      0.92      0.69        12
      D#:min       0.43      0.38      0.40         8
       D:maj       0.38      0.55      0.44      